# Claude 5.5 — Recupero V2 dopo interruzione del pilot (SOLO OFFLINE per default)

**Non rieseguire la run V1, non cancellare il claim e non aggiornare il checkout V1**. Il tentativo `openrca_claude_five_agents_development_v1__sonnet__cloudbed-1_q005__a0__r01` e' rimasto `CLAIM_PENDING_NO_AUTO_RETRY` dopo `KeyboardInterrupt` in `ToolNode`; non e' una diagnosi valida.

Questa nuova branch e' derivata dal commit V1 congelato `63a046723165976558840a30efb1b5097ad3fdf7` e cambia soltanto la gestione del warning KPI Pandas e i nomi/provenienza del protocollo. Mantiene cinque tool, otto regole, cinque varianti e planner 4+2, **nessun miglioramento di accuratezza viene rivendicato a priori**.

Fino alla cella pilot esplicitamente abilitata: **zero API Anthropic, zero scoring, zero modifiche ai file V1**.

In [ ]:
import os, subprocess, sys, hashlib, json, time
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DATA_V2 = Path('/content/drive/MyDrive/Tesi_OpenRCA')
REPO_V2 = Path('/content/Tesi_OpenRCA_Claude_V2')
BRANCH_V2 = 'feature/claude-clean-recovery-v2-2026-11-05'
REPO_URL = 'https://github.com/alessio1996A/Tesi_Laurea_Magistrale_OpenRCA.git'
FROZEN_V1_SHA = '63a046723165976558840a30efb1b5097ad3fdf7'
FROZEN_V1_PLAN_SHA = 'fb828dfcd7f2ac9e07b6d916ffd51e61b6a23052c1edbf7a06e965acb5a26158'

if not REPO_V2.exists():
    subprocess.run(['git','clone','--single-branch','--branch',
                    BRANCH_V2,REPO_URL,str(REPO_V2)],check=True)

def git_v2(*args):
    return subprocess.run(
        ['git','-C',str(REPO_V2),*args],check=True,
        text=True,capture_output=True
    ).stdout.strip()

assert git_v2('branch','--show-current') == BRANCH_V2
assert not git_v2('status','--porcelain'), 'Working tree V2 sporco: STOP'
COMMIT_V2 = git_v2('rev-parse','HEAD')
assert len(COMMIT_V2)==40

print('Branch V2:',BRANCH_V2)
print('Commit V2:',COMMIT_V2)
print('Directory separata:',REPO_V2)

subprocess.run([sys.executable,'-m','pip','install','-q',
                '-r',str(REPO_V2/'requirements.txt')],check=True)
if str(REPO_V2/'src') not in sys.path:
    sys.path.insert(0,str(REPO_V2/'src'))

from evaluation import claude_development_campaign as cdc_v2
assert cdc_v2.PROTOCOL == 'openrca_claude_five_agents_development_v2'
assert cdc_v2.BRANCH == BRANCH_V2
print('Protocollo V2 importato. Nessuna API chiamata.')

## Verifica di conservazione V1 — solo lettura

Confronta lo SHA del vecchio piano V1; verifica che il suo claim sia ancora presente e che nessun file di risultato sia stato creato. Se un file e' comparso successivamente, fermarsi per un audit umano. Non cancellare nulla.

In [ ]:
V1_PROTOCOL = 'openrca_claude_five_agents_development_v1'
V1_RUN = (
  V1_PROTOCOL+'__sonnet__cloudbed-1_q005__a0__r01'
)
v1_plan = DATA_V2/'data/protocols/v2'/f'{V1_PROTOCOL}_plan_v1.json'
assert hashlib.sha256(v1_plan.read_bytes()).hexdigest() == FROZEN_V1_PLAN_SHA
v1_dir = DATA_V2/'data/results/v2'/V1_PROTOCOL
v1_claim = v1_dir/(V1_RUN+'.claim.json')
v1_out = v1_dir/(V1_RUN+'.json')
v1_hash = v1_dir/(V1_RUN+'.sha256.json')
print('Piano V1 SHA: OK')
print('Claim V1:',v1_claim.exists())
print('Risultato V1:',v1_out.exists())
print('Hash V1:',v1_hash.exists())
assert v1_claim.exists(), 'Claim V1 non trovato: audit necessario'
assert not v1_out.exists() and not v1_hash.exists(), (
    'Stato V1 cambiato: STOP, esaminare prima di procedere'
)
print('Tentativo V1 preservato senza modifiche.')

## Test di regressione — zero API

Esegue prima il test specifico del warning KPI, poi l'intera suite del branch V2. Se qualsiasi test fallisce, non creare il piano e non chiamare Claude.

In [ ]:
env_offline = dict(os.environ)
env_offline['PYTHONPATH'] = str(REPO_V2/'src')
for k in ['ANTHROPIC_API_KEY','GEMINI_API_KEY','GOOGLE_API_KEY']:
    env_offline.pop(k, None)

for test_mask in ['test_pandas_kpi_warning_suppression.py','test_*.py']:
    print('\nESECUZIONE TEST:',test_mask)
    subprocess.run([sys.executable,'-m','unittest','discover',
        '-s',str(REPO_V2/'src/tests'),'-p',test_mask,'-v'],
        cwd=REPO_V2,env=env_offline,check=True)
print('TUTTA LA SUITE V2: OK. Nessuna API.')

## Confronto empirico dei tool: vecchio e nuovo — output identico

Esegue lo stesso **tool completo** delle metriche container sullo stesso incidente e lo stesso filtro regex con gruppi `(cpu|memory)`: una volta dal codice V1, una volta dal codice V2, in due processi separati senza chiavi API. Confronta SHA-256 del testo restituito. La correzione deve rimuovere i warning Pandas, **non alterare alcun carattere di evidenza**. Puo' richiedere decine di secondi per leggere due volte il CSV; nessuna chiamata Claude.

In [ ]:
OLD_REPO = Path('/content/Tesi_OpenRCA_Claude')
if not OLD_REPO.exists():
    subprocess.run(['git','clone','--single-branch','--branch',
       'feature/claude-cross-model-2026-11-05',REPO_URL,str(OLD_REPO)],
       check=True)
def old_git(*args):
    return subprocess.run(
        ['git','-C',str(OLD_REPO),*args],
        text=True,capture_output=True,check=True
    ).stdout.strip()

assert old_git('rev-parse','HEAD') == FROZEN_V1_SHA, (
   'Checkout V1 non coincide con il congelamento, STOP'
)
assert not old_git('status','--porcelain')

preview = cdc_v2.build_plan(DATA_V2, COMMIT_V2)
assert preview['run_count'] == 80
assert preview['recovery_provenance']['previous_plan_sha256'] == FROZEN_V1_PLAN_SHA
case = next(c for c in preview['cases'] if c['case_id']=='cloudbed-1_q005')

bench = r"""
import json,time,hashlib,warnings,sys
from agent.tools import tool_get_container_metrics
s,e,cb=int(sys.argv[1]),int(sys.argv[2]),sys.argv[3]
start=time.perf_counter()
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    output=tool_get_container_metrics(
       start_unix=s,end_unix=e,cmdb_id=None,
       kpi_name='(cpu|memory)',max_rows=10,
       market_system=cb
    )
report={
    'time_s':round(time.perf_counter()-start,3),
    'sha256':hashlib.sha256(output.encode('utf-8')).hexdigest(),
    'length':len(output),
    'warnings':len(w),
    'error':output.startswith('Errore'),
}
print(json.dumps(report,ensure_ascii=False))
"""

def benchmark(repo_path):
    env=dict(env_offline)
    env['PYTHONPATH']=str(repo_path/'src')
    p=subprocess.run(
        [sys.executable,'-c',bench,str(case['start_unix']),
         str(case['end_unix']),case['cloudbed']],
        cwd=repo_path,env=env,text=True,capture_output=True,timeout=90
    )
    if p.returncode:
        print(p.stderr[-1500:])
        raise RuntimeError('Benchmark fallito: '+str(repo_path))
    return json.loads(p.stdout.strip())

v1=benchmark(OLD_REPO)
v2=benchmark(REPO_V2)
print('V1:',v1)
print('V2:',v2)
assert not v1['error'] and not v2['error'], 'Tool ha restituito errore'
assert v1['sha256']==v2['sha256'], (
    'EVIDENZE DIVERSE: STOP, non creare piano'
)
assert v2['warnings']==0, 'Warning residui V2: STOP, da analizzare'
print('PARITA TOTALE DELL OUTPUT DEL TOOL: VERIFICATA')
print('NESSUNA API, NESSUNO SCORING, NESSUN FILE V1 MODIFICATO')

## Piano Claude V2 — preview e freeze create-only

**Solo se tutte le verifiche precedenti sono OK.** Non riutilizzare il piano V1. Il nuovo protocollo ha 80 nuovi identificativi V2 e annota il recupero del tentativo interrotto; A0 Sonnet `cloudbed-1_q005` sara' una nuova esecuzione, non un retry dello stesso run_id.

La preview qui sotto e' priva di chiamate API e predizioni.

In [ ]:
piano_v2 = cdc_v2.build_plan(DATA_V2, git_v2('rev-parse','HEAD'))
print('Protocollo:',piano_v2['protocol'])
print('Commit:',piano_v2['git_commit'])
print('Varianti:',piano_v2['variants'])
print('Totale:',piano_v2['run_count'])
print('Sonnet:',sum(r['model_key']=='sonnet' for r in piano_v2['runs']))
print('Haiku:',sum(r['model_key']=='haiku' for r in piano_v2['runs']))
print('Recupero originale:',piano_v2['recovery_provenance']['interrupted_run_id'])
print('Prima run nuova:',piano_v2['runs'][0]['run_id'])
assert piano_v2['runs'][0]['variant']=='A0'
assert piano_v2['runs'][0]['case_id']=='cloudbed-1_q005'
print('PREVIEW SOLO OFFLINE')

## Scrittura piano V2 — disattivata fino a verifica completa

**Attiva `CREA_PIANO_V2=True` solo se suite e parita' tool V1/V2 sono superate**, poi verifica che siano presenti esattamente 80 run ready. La scrittura `open('x')` non puo' sovrascrivere un piano esistente; non tocca i file V1.

In [ ]:
CREA_PIANO_V2 = False

dest = cdc_v2._plan_path(DATA_V2)
if CREA_PIANO_V2:
    if dest.exists():
        print('Piano V2 gia esistente: nessuna sovrascrittura')
    else:
        print('Piano V2 creato:', cdc_v2.write_plan(DATA_V2,REPO_V2))
else:
    print('Nessun piano creato.')
if dest.exists():
    pre = cdc_v2.preflight(DATA_V2,REPO_V2)
    print('SHA piano V2:', pre['plan_sha256'])
    print('Sonnet:',pre['models']['sonnet'])
    print('Haiku:',pre['models']['haiku'])
else:
    print('Piano V2 assente (previsto se CREA_PIANO_V2=False).')
print('Nessuna API chiamata')

## Verifica Secret Anthropic senza inferenza

Apri Secrets in Colab e consenti a questo notebook di leggere `ANTHROPIC_API_KEY`. Non scrivere mai la chiave su GitHub o in chat. Questa cella la copia nell'ambiente runtime, senza stamparla e **senza chiamare il provider**.

In [ ]:
from google.colab import userdata

anthropic_secret=userdata.get('ANTHROPIC_API_KEY')
assert anthropic_secret and anthropic_secret.strip()
os.environ['ANTHROPIC_API_KEY']=anthropic_secret
del anthropic_secret
print('Secret ANTHROPIC_API_KEY disponibile: SI, valore nascosto')
print('Nessuna API chiamata')

## Nuova prima run Sonnet A0 V2 — OFF per default

Questo e' un nuovo identificativo `development_v2`, non e' il tentativo interrotto V1. Attiva solo dopo la verifica di tutto il preflight, la creazione del piano V2 e l'autorizzazione esplicita di una sola nuova run. `BUDGET_USD` e' riferimento, **non limite rigido**. **Non eseguire se esistono altre run V2 tentate o se il notebook e' gia' in esecuzione.**

E' normale che l'agente resti impegnato diversi secondi a ogni chiamata ai tool; il solo warning non richiede di interromperlo. Se la cella termina per interruzione o timeout, esegui esclusivamente la cella successiva di audit; **mai ripetere la stessa run**.

In [ ]:
AVVIA_NUOVO_PILOTA_V2 = False  # cambia solo dopo conferma e preflight
BUDGET_USD = 0.50  # costo indicativo singola run, NON hard cap

if not AVVIA_NUOVO_PILOTA_V2:
    print('Nessuna API chiamata. Pilota V2 DISABILITATO.')
else:
    status = cdc_v2.preflight(DATA_V2,REPO_V2)
    assert status['models']['sonnet']=={'ready':40}
    assert status['models']['haiku']=={'ready':40}
    assert status['runs'][0]['variant']=='A0'
    assert status['runs'][0]['case_id']=='cloudbed-1_q005'
    print('Avvio una sola run SONNET A0 V2:')
    print(status['runs'][0]['run_id'])
    answer=cdc_v2.run_batch(
       DATA_V2,REPO_V2,model_key='sonnet',
       max_new_runs=1,confirm_api=True,
       approved_budget_usd=BUDGET_USD
    )
    print('Esito:',answer)


## Audit read-only dopo pilot o interruzione

Questa cella puo' essere eseguita sempre. Non chiama Claude, non cancella claim, non crea freeze o scoring.

In [ ]:
p=cdc_v2._plan_path(DATA_V2)
if p.exists():
    state=cdc_v2.preflight(DATA_V2,REPO_V2)
    first=state['runs'][0]
    claim,result,digest=cdc_v2._paths(DATA_V2,first['run_id'])
    print('Run:',first['run_id'])
    print('Stato:',first['status'])
    print('Claim:',claim.exists(),'JSON:',result.exists(),
          'SHA:',digest.exists())
    print('Sonnet:',state['models']['sonnet'])
    print('Haiku:',state['models']['haiku'])
    if result.exists():
        data=json.loads(result.read_text(encoding='utf-8'))
        print('JSON status:',data.get('json_status'))
        print('Errore:',str(data.get('error'))[:500])
        print('Metriche:',data.get('metrics'))
else:
    print('Piano V2 non ancora creato.')
print('SOLO LETTURA: Nessuna API chiamata')